# 02_2 - Enriquecimiento con cotización del dólar BNA

## Objetivo
Agregar al dataset limpio la **cotización del dólar BNA (precio Venta)** correspondiente a cada aviso,
según su fecha de scraping (`scrap_date`).

Como las cotizaciones no coinciden exactamente con las fechas de scraping, se usa `merge_asof`
para asignar la **última cotización disponible igual o anterior** a cada fecha.

## Entradas
- `outputs/interm/02_df_limpio_base.csv` — dataset limpio del paso 02
- `cotizacion_dolar_bna.csv` — historial de cotizaciones del dólar BNA

## Salidas
- `outputs/interm/02_2_df_con_dolar.csv` — dataset enriquecido con columna `dolar_venta`
- `outputs/tablas/02_2_reporte_dolar.csv` — reporte de cobertura del join


## Parámetros y librerías

In [1]:
from pathlib import Path
import pandas as pd

# Paths
out_interm = Path('outputs/interm')
out_tablas = Path('outputs/tablas')

# Crear carpetas si no existen
out_interm.mkdir(parents=True, exist_ok=True)
out_tablas.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

## 1. Cargar el dataset limpio (output del paso 02)

In [2]:
in_path = out_interm / '02_df_limpio_base.csv'
df = pd.read_csv(in_path)

print(f'Filas cargadas: {len(df)}')
print(f'Columnas: {df.shape[1]}')
print(f'\nPrimeras filas de scrap_date:')
print(df['scrap_date'].head(10).to_string())

Filas cargadas: 104084
Columnas: 33

Primeras filas de scrap_date:
0    2024-09-23 08:28:14.272951
1    2024-09-23 08:28:14.272951
2    2024-09-23 08:28:14.272951
3    2024-09-23 08:28:14.272951
4    2024-09-23 08:28:14.272951
5    2024-09-23 08:28:14.272951
6    2024-09-23 08:28:14.272951
7    2024-09-23 08:28:14.272951
8    2024-09-23 08:28:14.272951
9    2024-09-23 08:28:14.272951


## 2. Ver fechas únicas de scraping

Son pocas fechas distintas (el scraping se hizo en pocas sesiones), lo que hace viable este enriquecimiento.

In [4]:
# Convertir scrap_date a datetime
df['scrap_date'] = pd.to_datetime(df['scrap_date'], errors='coerce')

# Extraer solo la fecha (sin hora) para el join
df['fecha'] = df['scrap_date'].dt.normalize()

print('Fechas únicas de scraping en el dataset:')
fechas_unicas = sorted(df['fecha'].dropna().unique())
print(fechas_unicas)
print(f'\nTotal fechas distintas: {len(fechas_unicas)}')
print(f'NaT en scrap_date: {df["scrap_date"].isna().sum()}')

Fechas únicas de scraping en el dataset:
[Timestamp('2024-09-23 00:00:00'), Timestamp('2024-09-24 00:00:00'), Timestamp('2024-09-29 00:00:00'), Timestamp('2024-09-30 00:00:00'), Timestamp('2025-03-09 00:00:00'), Timestamp('2025-07-12 00:00:00')]

Total fechas distintas: 6
NaT en scrap_date: 0


## 3. Cargar el CSV de cotizaciones del dólar BNA

> **Formato esperado del CSV:**  
> Separador `;`, columnas `Fecha cotizacion` (DD/MM/AAAA) y `Venta` (número decimal).
>
> Ejemplo:
> ```
> Fecha cotizacion;Compra;Venta
> 23/09/2024;955.00;975.00
> 24/09/2024;957.00;977.00
> ```

In [5]:
dfDolar = pd.read_csv(
    'cotizacion_dolar_bna.csv',
    sep=';',
    decimal='.'
)

print('Columnas del CSV dólar:', dfDolar.columns.tolist())
print(f'Filas cargadas: {len(dfDolar)}')
print('\nPrimeras filas:')
display(dfDolar.head())

Columnas del CSV dólar: ['Fecha cotizacion', 'Compra', 'Venta', 'Unnamed: 3']
Filas cargadas: 572

Primeras filas:


,Fecha cotizacion,Compra,Venta,Unnamed: 3
0,2/1/2024,"790,2500","830,2500",NaN
1,3/1/2024,"791,0000","831,0000",NaN
2,4/1/2024,"791,5000","831,5000",NaN
3,5/1/2024,"792,0000","832,0000",NaN
4,8/1/2024,"793,5000","833,5000",NaN


## 4. Preparar el dataframe de cotizaciones

In [6]:
# Convertir fecha (formato argentino: DD/MM/AAAA)
dfDolar['Fecha cotizacion'] = pd.to_datetime(
    dfDolar['Fecha cotizacion'],
    dayfirst=True,   # <-- importante: el día va primero (formato AR)
    errors='coerce'
)

# Eliminar filas con fecha inválida
n_antes = len(dfDolar)
dfDolar = dfDolar.dropna(subset=['Fecha cotizacion'])
n_despues = len(dfDolar)
print(f'Filas eliminadas por fecha inválida: {n_antes - n_despues}')

# Quedarse solo con las columnas necesarias
dfDolar = dfDolar[['Fecha cotizacion', 'Venta']].copy()

# Renombrar para mayor claridad en el dataset final
dfDolar = dfDolar.rename(columns={'Venta': 'dolar_venta'})

print(f'\nRango de cotizaciones disponibles:')
print(f'  Desde: {dfDolar["Fecha cotizacion"].min().date()}')
print(f'  Hasta: {dfDolar["Fecha cotizacion"].max().date()}')
print(f'\nCotizaciones que se usarán para las fechas de scraping:')
for f in fechas_unicas:
    # Buscar manualmente la cotización correspondiente (solo para mostrar)
    match = dfDolar[dfDolar['Fecha cotizacion'] <= f].tail(1)
    if len(match):
        print(f'  {str(f)[:10]} → cotización del {str(match["Fecha cotizacion"].values[0])[:10]}: ${match["dolar_venta"].values[0]}')
    else:
        print(f'  {str(f)[:10]} → ⚠️  sin cotización disponible')

Filas eliminadas por fecha inválida: 0

Rango de cotizaciones disponibles:
  Desde: 2024-01-02
  Hasta: 2026-05-14

Cotizaciones que se usarán para las fechas de scraping:
  2024-09-23 → cotización del 2024-09-23: $985,5000
  2024-09-24 → cotización del 2024-09-24: $986,0000
  2024-09-29 → cotización del 2024-09-27: $989,5000
  2024-09-30 → cotización del 2024-09-30: $990,0000
  2025-03-09 → cotización del 2025-03-07: $1086,0000
  2025-07-12 → cotización del 2025-07-11: $1275,0000


## 5. Join con `merge_asof`

`merge_asof` une las dos tablas buscando, para cada fila del dataset principal,
la cotización **más reciente disponible** (igual o anterior a la fecha de scraping).

Ambas tablas deben estar ordenadas por fecha antes del join.

In [7]:
# Ordenar ambas tablas por fecha (obligatorio para merge_asof)
df = df.sort_values('fecha').reset_index(drop=True)
dfDolar = dfDolar.sort_values('Fecha cotizacion').reset_index(drop=True)

# Join: para cada aviso, busca la cotización <= fecha de scraping
df_enriq = pd.merge_asof(
    df,
    dfDolar,
    left_on='fecha',
    right_on='Fecha cotizacion',
    direction='backward'   # <-- usa la cotización más reciente hacia atrás
)

print(f'Filas después del join: {len(df_enriq)}')
print(f'\nCobertura de dolar_venta:')
n_ok = df_enriq['dolar_venta'].notna().sum()
n_nulo = df_enriq['dolar_venta'].isna().sum()
pct_ok = round(100 * n_ok / len(df_enriq), 2)
print(f'  Con cotización asignada: {n_ok} ({pct_ok}%)')
print(f'  Sin cotización (NaN):    {n_nulo}')

Filas después del join: 104084

Cobertura de dolar_venta:
  Con cotización asignada: 104084 (100.0%)
  Sin cotización (NaN):    0


## 6. Verificar el resultado

In [8]:
# Ver cotizaciones asignadas por fecha de scraping
print('Cotización asignada por fecha de scraping:')
resumen = (
    df_enriq
    .groupby(['fecha', 'Fecha cotizacion', 'dolar_venta'])
    .size()
    .reset_index(name='cantidad_avisos')
)
display(resumen)

# Muestra de columnas clave
print('\nMuestra de columnas clave en el dataset enriquecido:')
cols_muestra = ['scrap_date', 'fecha', 'Fecha cotizacion', 'dolar_venta', 'price_val', 'currency']
cols_disponibles = [c for c in cols_muestra if c in df_enriq.columns]
display(df_enriq[cols_disponibles].drop_duplicates('fecha').head(10))

Cotización asignada por fecha de scraping:


,fecha,Fecha cotizacion,dolar_venta,cantidad_avisos
0,2024-09-23,2024-09-23,"985,5000",13624
1,2024-09-24,2024-09-24,"986,0000",27288
2,2024-09-29,2024-09-27,"989,5000",13399
3,2024-09-30,2024-09-30,"990,0000",14564
4,2025-03-09,2025-03-07,"1086,0000",27725
5,2025-07-12,2025-07-11,"1275,0000",7484



Muestra de columnas clave en el dataset enriquecido:


,scrap_date,fecha,Fecha cotizacion,dolar_venta,price_val,currency
0,2024-09-23 08:28:14.272951,2024-09-23,2024-09-23,"985,5000",500.0,USD
13624,2024-09-24 14:20:03.244872,2024-09-24,2024-09-24,"986,0000",270000.0,USD
40912,2024-09-29 23:47:13.054893,2024-09-29,2024-09-27,"989,5000",580000.0,ARS
54311,2024-09-30 01:10:57.449573,2024-09-30,2024-09-30,"990,0000",368200.0,USD
68875,2025-03-09 02:02:55.634007,2025-03-09,2025-03-07,"1086,0000",82400.0,USD
96600,2025-07-12 13:58:06.735039,2025-07-12,2025-07-11,"1275,0000",830000.0,USD


## 7. Exportar resultados

In [9]:
# Reporte de cobertura
reporte = pd.DataFrame({
    'metrica': [
        'filas_input',
        'filas_output',
        'con_cotizacion',
        'sin_cotizacion',
        'pct_cobertura'
    ],
    'valor': [
        len(df),
        len(df_enriq),
        n_ok,
        n_nulo,
        pct_ok
    ]
})

# Exportar dataset enriquecido
out_dataset = out_interm / '02_2_df_con_dolar.csv'
df_enriq.to_csv(out_dataset, index=False)

# Exportar reporte
out_reporte = out_tablas / '02_2_reporte_dolar.csv'
reporte.to_csv(out_reporte, index=False)

display(reporte)
print('\nGenerado:', out_dataset)
print('Generado:', out_reporte)

,metrica,valor
0,filas_input,104084.0
1,filas_output,104084.0
2,con_cotizacion,104084.0
3,sin_cotizacion,0.0
4,pct_cobertura,100.0



Generado: outputs/interm/02_2_df_con_dolar.csv
Generado: outputs/tablas/02_2_reporte_dolar.csv
